In [1]:
# Parameters
base_path = "C:\\Users\\ander\\OneDrive\\TCC_USP"
run_id = "20251119_133015"


In [2]:
# 1. Setup de caminhos locais
import os
import subprocess
from datetime import datetime
from pathlib import Path

from src.io import paths
from src.config import loader as cfg

DATA_PATHS = paths.get_data_paths()
PROJECT_PATHS = paths.get_project_paths()
BASE_PATH = str(DATA_PATHS["base"])
RAW_PATH = str(DATA_PATHS["data_raw"])
PROC_PATH = str(DATA_PATHS["data_processed"])
INTERIM_PATH = str(DATA_PATHS["data_interim"])
CONFIG = cfg.load_config()

NB_NAME = "09_lstm_real"
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

print("BASE_PATH:", BASE_PATH)
print("PROC_PATH:", PROC_PATH)


BASE_PATH: C:\Users\ander\OneDrive\TCC_USP
PROC_PATH: C:\Users\ander\OneDrive\TCC_USP\data_processed


In [3]:
# 2. Carregar dados
import pandas as pd, numpy as np

ibov_file = os.path.join(PROC_PATH, "ibovespa_clean.csv")
news_file = os.path.join(PROC_PATH, "noticias_real_clean.csv")

df_ibov = pd.read_csv(ibov_file)
df_news = pd.read_csv(news_file)

if "date" in df_ibov.columns: df_ibov.rename(columns={"date":"data"}, inplace=True)
if "date" in df_news.columns: df_news.rename(columns={"date":"data"}, inplace=True)

df_ibov["data"] = pd.to_datetime(df_ibov["data"])
df_news["data"] = pd.to_datetime(df_news["data"])

if "clean_text" not in df_news.columns:
    df_news["clean_text"] = df_news.get("texto","").fillna("")

print("IBOV:", df_ibov.shape, "| NEWS:", df_news.shape)

IBOV: (20, 2) | NEWS: (518, 6)


In [4]:
# 3. Merge notícias agregadas + IBOV
df_text = df_news.groupby("data", as_index=False)["clean_text"].apply(
    lambda s: " ".join([str(x) for x in s if str(x).strip()])
)

df_ibov = df_ibov.sort_values("data").reset_index(drop=True)
df_ibov["ret1"] = df_ibov["close"].pct_change().shift(-1)
df_ibov["y"] = (df_ibov["ret1"] > 0).astype(int)
df_target = df_ibov.dropna(subset=["ret1"]).copy()

df = pd.merge(df_target[["data","close","y"]], df_text, on="data", how="inner").sort_values("data")

# fallback dummy
if df.shape[0] == 0:
    print("⚠️ Nenhuma interseção → dataset dummy criado.")
    df = pd.DataFrame({
        "data": pd.date_range("2024-01-02", periods=20, freq="D"),
        "close": np.linspace(130000,132000,20),
        "y": [0,1,0,1,1,0,1,0,1,1,0,1,0,1,1,0,1,0,1,1],
        "clean_text": [
            "mercado alta otimismo",
            "queda dolar pressao",
            "petrobras avanco dividendos",
            "ibovespa estabilidade noticias",
            "crescimento economia brasil",
            "inflacao preocupa investidores",
            "mercado futuro reage positivo",
            "politica monetaria em foco",
            "bancos puxam queda ibovespa",
            "investidores cautela incerteza",
            "petroleo alta favorece petrobras",
            "ibovespa sobe com fluxo estrangeiro",
            "corte juros anima mercado",
            "dolar pressiona exportadoras",
            "economia americana influencia b3",
            "acoes blue chips sustentam alta",
            "queda commodities preocupa",
            "lucros corporativos animam investidores",
            "mercado expectativa reformas",
            "ibovespa encerra semana em alta"
        ]
    })

print("Dataset final:", df.shape)
display(df.head())

⚠️ Nenhuma interseção → dataset dummy criado.
Dataset final: (20, 4)


,data,close,y,clean_text
0,2024-01-02,130000.000000,0,mercado alta otimismo
1,2024-01-03,130105.263158,1,queda dolar pressao
2,2024-01-04,130210.526316,0,petrobras avanco dividendos
3,2024-01-05,130315.789474,1,ibovespa estabilidade noticias
4,2024-01-06,130421.052632,1,crescimento economia brasil


In [5]:
# 4. Vetorização com embeddings
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(df["clean_text"].fillna("").tolist(), show_progress_bar=True)

X = np.array(embeddings)
y = df["y"].reset_index(drop=True)
print("Embeddings gerados:", X.shape)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings gerados: (20, 384)


In [6]:
# Dependência já deve estar instalada via requirements.txt

In [7]:
# 5. Transformar em sequências para LSTM
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

def create_sequences(X, y, window=3):
    X_seq, y_seq = [], []
    for i in range(len(X) - window):
        X_seq.append(X[i:i+window])
        y_seq.append(y[i+window])
    return np.array(X_seq), np.array(y_seq)

X_seq, y_seq = create_sequences(X, y, window=3)

print("X_seq:", X_seq.shape, "| y_seq:", y_seq.shape)

X_seq: (17, 3, 384) | y_seq: (17,)


In [8]:
# 6. Construir modelo LSTM
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

timesteps, features = X_seq.shape[1], X_seq.shape[2]

model = Sequential([
    LSTM(64, input_shape=(timesteps, features)),
    Dropout(0.3),
    Dense(32, activation="relu"),
    Dropout(0.2),
    Dense(1, activation="sigmoid")
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

C:\Users\ander\AppData\Roaming\Python\Python311\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                          │ (None, 64)                  │         114,944 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 117,057 (457.25 KB)

 Trainable params: 117,057 (457.25 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
# 7. Treinar e avaliar
from sklearn.metrics import roc_auc_score, accuracy_score

split = int(0.8 * len(X_seq))
X_train, X_test = X_seq[:split], X_seq[split:]
y_train, y_test = y_seq[:split], y_seq[split:]

history = model.fit(X_train, y_train, validation_split=0.2,
                    epochs=20, batch_size=8, verbose=1)

y_pred_proba = model.predict(X_test).ravel()
y_pred = (y_pred_proba > 0.5).astype(int)

auc = roc_auc_score(y_test, y_pred_proba) if len(np.unique(y_test)) > 1 else float("nan")
mda = accuracy_score(y_test, y_pred)

print(f"\nResultados LSTM: AUC={auc:.3f}, MDA={mda:.3f}")

Epoch 1/20


1/2 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step - accuracy: 0.7500 - loss: 0.6850

2/2 ━━━━━━━━━━━━━━━━━━━━ 5s 643ms/step - accuracy: 0.7000 - loss: 0.6855 - val_accuracy: 0.6667 - val_loss: 0.6718


Epoch 2/20


1/2 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.6250 - loss: 0.6787

...

1/2 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.6250 - loss: 0.6654

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step - accuracy: 0.6000 - loss: 0.6707 - val_accuracy: 0.6667 - val_loss: 0.6510


Epoch 6/20


1/2 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.6250 - loss: 0.6620


KeyboardInterrupt



In [ ]:
# 6. Organizar resultados e salvar em JSON (para o dashboard)
import json, os
import numpy as np

nb_name = "09_lstm_real"

# Garante que aucs e mdas existam (mesmo se não definidos antes)
if "aucs" not in locals():
    aucs = []
if "mdas" not in locals():
    mdas = []

# Transformar métricas da LSTM em dict compatível
results = {
    "LSTM": {
        "AUC": float(np.mean(aucs)) if len(aucs) > 0 else 0.0,
        "MDA": float(np.mean(mdas)) if len(mdas) > 0 else 0.0
    }
}

# results -> dict serializável
results_dict = {k: {"AUC": float(v["AUC"]), "MDA": float(v["MDA"])} for k, v in results.items()}

# Salvar JSON em data_processed
out_file = os.path.join(PROC_PATH, f"results_{nb_name}.json")
with open(out_file, "w") as f:
    json.dump(results_dict, f, indent=4)

print(f"✅ Resultados salvos em {out_file}")